## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Persiapan SparkSession dan Folder

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType
)
from pyspark.sql.functions import col, sum as spark_sum, count, window
import pandas as pd
import os
import shutil
import time

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

print("SparkSession dan folder streaming siap.")

26/11/05 08:01:30 WARN Utils: Your hostname, angger resolves to a loopback address: 127.0.1.1; using 10.90.66.217 instead (on interface wlp3s0)
26/11/05 08:01:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 08:01:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession dan folder streaming siap.


**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

In [2]:
skema_pesanan = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True),
])

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_tugas)

print("Apakah DataFrame bersifat streaming?", df_stream.isStreaming)

Apakah DataFrame bersifat streaming? True


**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

In [3]:
df_pesanan = df_stream.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_cabang = df_pesanan.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

query1 = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query agregasi cabang aktif:", query1.isActive)

Query agregasi cabang aktif: True


**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

In [4]:
# Sumber yang sama dapat dibaca oleh query streaming yang berbeda.
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_pesanan) \
    .option("header", True) \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .load(folder_tugas)

df_pesanan_waktu = df_stream_waktu.withColumn(
    "total_harga",
    col("qty") * col("harga_satuan")
)

agregasi_waktu = df_pesanan_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan")
)

query2 = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query agregasi window aktif:", query2.isActive)

Query agregasi window aktif: True


**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

In [5]:
def kirim_batch(nama_file, data):
    pd.DataFrame(data).to_csv(
        os.path.join(folder_tugas, nama_file),
        index=False
    )
    print(f"{nama_file} berhasil dikirim.")
    time.sleep(6)
    spark.sql("""
        SELECT nama_cabang, total_pendapatan, jumlah_pesanan
        FROM agregasi_cabang
        ORDER BY nama_cabang
    """).show(truncate=False)

waktu_batch = "2026-11-05 08:00:00"

# Batch 1
kirim_batch("batch_1.csv", {
    "pesanan_id": ["P001", "P002", "P003"],
    "nama_cabang": ["Magelang", "Yogyakarta", "Magelang"],
    "nama_menu": ["Kopi Susu", "Americano", "Matcha"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 15000, 22000],
    "waktu_pesan": [waktu_batch] * 3
})

# Batch 2
kirim_batch("batch_2.csv", {
    "pesanan_id": ["P004", "P005", "P006"],
    "nama_cabang": ["Solo", "Magelang", "Yogyakarta"],
    "nama_menu": ["Cappuccino", "Latte", "Kopi Hitam"],
    "qty": [2, 2, 1],
    "harga_satuan": [25000, 20000, 12000],
    "waktu_pesan": [waktu_batch] * 3
})

# Batch 3
kirim_batch("batch_3.csv", {
    "pesanan_id": ["P007", "P008", "P009"],
    "nama_cabang": ["Solo", "Yogyakarta", "Magelang"],
    "nama_menu": ["Mocha", "Es Kopi", "Teh"],
    "qty": [1, 3, 2],
    "harga_satuan": [27000, 18000, 10000],
    "waktu_pesan": [waktu_batch] * 3
})

batch_1.csv berhasil dikirim.


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|Magelang   |102000          |2             |
|Yogyakarta |15000           |1             |
+-----------+----------------+--------------+

batch_2.csv berhasil dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|Magelang   |142000          |3             |
|Solo       |50000           |1             |
|Yogyakarta |27000           |2             |
+-----------+----------------+--------------+

batch_3.csv berhasil dikirim.
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|Magelang   |162000          |4             |
|Solo       |77000           |2             |
|Yogyakarta |81000           |3             |
+-----------+----------------+--------------+



### Periksa hasil agregasi window

In [6]:
time.sleep(3)
spark.sql("""
    SELECT window.start, window.end, total_pendapatan
    FROM agregasi_waktu
    ORDER BY window.start
""").show(truncate=False)

+-------------------+-------------------+----------------+
|start              |end                |total_pendapatan|
+-------------------+-------------------+----------------+
|2026-11-05 08:00:00|2026-11-05 08:02:00|320000          |
+-------------------+-------------------+----------------+



**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

In [7]:
folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

# Sink ketiga menyimpan data mentah, tanpa agregasi.
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

# Kirim batch tambahan setelah query Parquet aktif.
pd.DataFrame({
    "pesanan_id": ["P010", "P011"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Caramel Latte", "Matcha"],
    "qty": [2, 1],
    "harga_satuan": [23000, 22000],
    "waktu_pesan": ["2026-11-05 08:01:00", "2026-11-05 08:01:00"]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

time.sleep(8)

# Verifikasi file hasil sink Parquet.
print("Isi folder output:")
print(os.listdir(folder_output))

df_parquet = spark.read.parquet(folder_output)
df_parquet.show(truncate=False)
print("Jumlah data mentah tersimpan:", df_parquet.count())

Isi folder output:
['.part-00001-d82fe36a-0c0e-48a1-9433-62b91269050e-c000.snappy.parquet.crc', '_spark_metadata', 'part-00000-1f31e06a-990b-4652-8666-b7c7652bcba2-c000.snappy.parquet', '.part-00000-1f31e06a-990b-4652-8666-b7c7652bcba2-c000.snappy.parquet.crc', 'part-00003-bd6a3d80-950f-49cc-b0b5-1ea247f9bad4-c000.snappy.parquet', 'part-00001-d82fe36a-0c0e-48a1-9433-62b91269050e-c000.snappy.parquet', '.part-00002-e0e4fa37-b7ba-453e-b787-d5573806787a-c000.snappy.parquet.crc', 'part-00002-e0e4fa37-b7ba-453e-b787-d5573806787a-c000.snappy.parquet', '.part-00003-bd6a3d80-950f-49cc-b0b5-1ea247f9bad4-c000.snappy.parquet.crc']
+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu    |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------+---+------------+-------------------+
|P004      |Solo       |Cappuccino   |2  |25000       |2026-11-05 08:00:00|
|P005      |Magelang   |Latte        |2  |20000       |2026-11-05 08:

### Menghentikan seluruh query

In [8]:
query1.stop()
query2.stop()
query3.stop()

print("Status query 1:", query1.isActive)
print("Status query 2:", query2.isActive)
print("Status query 3:", query3.isActive)

spark.stop()
print("Seluruh streaming query dihentikan dan SparkSession ditutup.")

Status query 1: False
Status query 2: False
Status query 3: False
Seluruh streaming query dihentikan dan SparkSession ditutup.
